In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

output_width = 600
output_height = 800

'''=====圖片路徑更改====='''
image_path = "../Data/quiz3/skew.jpg"
image = cv2.imread(image_path)

In [2]:
# 遇到照片太大，沒辦法手動點選四角
# 縮小顯示，讓大照片也能完整放進視窗
# 之後標點要記得放大回去才是對的
h, w = image.shape[:2]
scale = min(1.0, 1100 / w, 750 / h)

display = cv2.resize(
    image,
    (round(w * scale), round(h * scale))
)

# 用實際縮放比例換回原圖座標
scale_x = display.shape[1] / w
scale_y = display.shape[0] / h

In [3]:
clicked = []
window_name = "Select TL, TR, BR, BL | R: reset | Enter: confirm | Esc: cancel"

def on_mouse(event, x, y, flags, param):
    if event == cv2.EVENT_LBUTTONDOWN and len(clicked) < 4:
        clicked.append((x, y))

cv2.namedWindow(window_name, cv2.WINDOW_AUTOSIZE)
cv2.setMouseCallback(window_name, on_mouse)

confirmed = False

print("依物體正視時的方向，依序點選：左上 → 右上 → 右下 → 左下")
print("R：重新選取；Enter：確認；Esc：取消")

依物體正視時的方向，依序點選：左上 → 右上 → 右下 → 左下
R：重新選取；Enter：確認；Esc：取消


In [4]:
try:
    while True:
        preview = display.copy()

        for i, (x, y) in enumerate(clicked):
            cv2.circle(preview, (x, y), 5, (0, 0, 255), -1)
            cv2.putText(
                preview, ["TL", "TR", "BR", "BL"][i],
                (x + 8, y + 8),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.6, (0, 0, 255), 2
            )
        
        cv2.imshow(window_name, preview)
        key = cv2.waitKey(20) & 0xFF

        if key == ord("r"):
            clicked.clear()
        elif key in (10, 13) and len(clicked) == 4:
            confirmed = True
            break
        elif key == 27:
            break

        if cv2.getWindowProperty(window_name, cv2.WND_PROP_VISIBLE) < 1:
            break
finally:
    cv2.destroyAllWindows()

if not confirmed:
    raise RuntimeError("已取消，尚未完成四角選取")

In [5]:
# ===== 換回原圖座標 =====
src = np.array(clicked, dtype=np.float32)
src[:, 0] /= scale_x
src[:, 1] /= scale_y

# 確認四點依邊界排列，沒有交叉或退化
if not cv2.isContourConvex(src) or cv2.contourArea(src) < 25:
    raise ValueError("四角排列不正確，請依左上、右上、右下、左下重新選取")

# ===== 透視轉換 =====
# OpenCV 的點座標與 NumPy 的影像索引，順序不同
# OpenCV: (200, 100) = numpy (100, 200)，numpy是正常的row column index。
# OpenCV從原點向右：(0,0) -> (100,0)
dst = np.float32([
    [0, 0],
    [output_width - 1, 0],
    [output_width - 1, output_height - 1],
    [0, output_height - 1]
])

matrix = cv2.getPerspectiveTransform(src, dst)
corrected = cv2.warpPerspective(
    image, matrix, (output_width, output_height)
)

In [6]:
'''=====儲存校正後影像路徑====='''
cv2.imwrite("result.png", corrected)

True